In [ ]:
%py
# -- PySpark code to mask the last 4 digits of invoice_number in the d_product_revenue_clone table

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit, substring
from pyspark.sql.types import StringType

# -- Initialize Spark session
spark = SparkSession.builder.appName("MaskInvoiceNumber").getOrCreate()

try:
    # -- Drop the d_product_revenue_clone table if exists
    spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone")

    # -- Create a replica of d_product_revenue table
    spark.sql("""
        CREATE TABLE purgo_playground.d_product_revenue_clone AS
        SELECT * FROM purgo_playground.d_product_revenue
    """)

    # -- Load the replica table into a DataFrame
    df_clone = spark.table("purgo_playground.d_product_revenue_clone")

    # -- Mask the last 4 digits of invoice_number by converting it to STRING
    masked_df = df_clone.withColumn(
        "invoice_number",
        concat(substring(col("invoice_number").cast(StringType()), 1, 6), lit("****"))
    )

    # -- Write the DataFrame back to the clone table
    masked_df.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")

except Exception as e:
    # -- Capture and log any errors that occur during processing
    print(f"Error during masking process: {e}")

# -- Stop the Spark session
spark.stop()